In [ ]:
import os
import shutil
import time
import subprocess
import sys
import re
from pathlib import Path

In [ ]:
# --- Setup & Dependencies ---
def run_bootstrap_command(cmd, label):
    print(f"[SETUP] {label}")
    subprocess.run(cmd, check=True)

def ensure_system_tools():
    missing = [tool for tool in ["ffmpeg"] if shutil.which(tool) is None]
    if missing:
        run_bootstrap_command(["apt-get", "update", "-qq"], "Preparing Colab package index")
        run_bootstrap_command(["apt-get", "install", "-y", "ffmpeg"], "Installing ffmpeg")

def ensure_python_package(import_name, package_name=None):
    package_name = package_name or import_name
    try:
        __import__(import_name)
    except ModuleNotFoundError:
        run_bootstrap_command([sys.executable, "-m", "pip", "install", "-q", package_name], "Installing " + package_name)

ensure_system_tools()
ensure_python_package("yt_dlp", "yt-dlp")
ensure_python_package("ipywidgets")

import ipywidgets as widgets
from IPython.display import display, clear_output
from google.colab import drive
from yt_dlp import YoutubeDL

In [ ]:
# --- Core Application ---
class YouTubeSpecialistDL:
    def __init__(self):
        self.drive_mount_path = Path("/content/drive")
        self.my_drive_path = self.drive_mount_path / "MyDrive"
        self.youtube_base_dir = self.my_drive_path / "YouTubeDownloads"
        self.temp_dir = Path("/content/YT-Temp")

    def mount_google_drive(self):
        if not self.my_drive_path.exists():
            print("Mounting Google Drive...")
            drive.mount(str(self.drive_mount_path))
        else:
            print("Google Drive is already mounted.")

    def sanitize_name(self, name):
        name = str(name or "").strip()
        name = re.sub(r'[\/\\:*?"<>|]', "_", name)
        return name

    def force_drive_sync(self):
        try:
            subprocess.run(["sync"], check=False)
        except Exception:
            pass
        time.sleep(2)

    def is_temporary_output_file(self, path):
        path = Path(path)
        name = path.name.lower()
        suffix = path.suffix.lower()
        if suffix in [".part", ".ytdl", ".temp", ".tmp"]: return True
        if name.endswith(".part") or name.endswith(".ytdl") or name.endswith(".tmp"): return True
        return False

    def collect_final_files(self, folder):
        folder = Path(folder)
        if not folder.exists(): return []
        files = []
        for item in folder.glob("**/*"):
            if item.is_file() and not self.is_temporary_output_file(item):
                try:
                    if item.stat().st_size > 0: files.append(item)
                except Exception:
                    pass
        return files

    def transfer_folder_contents_to_drive(self, source_dir, destination_dir):
        source_dir = Path(source_dir)
        destination_dir = Path(destination_dir)
        destination_dir.mkdir(parents=True, exist_ok=True)

        files = self.collect_final_files(source_dir)
        transferred = []

        for item in files:
            target = destination_dir / item.name
            counter = 1
            while target.exists():
                target = destination_dir / f"{item.stem}_{counter}{item.suffix}"
                counter += 1

            shutil.copy2(item, target)
            transferred.append(target)
            item.unlink()

        self.force_drive_sync()
        return transferred

    def get_format_config(self, dl_type, quality):
        res_limit = "" if quality == "Best" else f"[height<={quality.replace('p', '')}]"

        if dl_type == "Audio Only":
            format_str = "bestaudio/best"
            postprocessors = [{
                'key': 'FFmpegExtractAudio',
                'preferredcodec': 'mp3',
                'preferredquality': '192',
            }]
        elif dl_type == "Video Only":
            format_str = f"bestvideo{res_limit}[ext=mp4]/bestvideo{res_limit}/bestvideo"
            postprocessors = []
        else:  # Video + Audio
            format_str = f"bestvideo{res_limit}[ext=mp4]+bestaudio[ext=m4a]/best{res_limit}[ext=mp4]/best{res_limit}/best"
            postprocessors = []

        return format_str, postprocessors

    def extract_downloaded_ids_from_archive(self, archive_file):
        if not archive_file.exists():
            return set()

        archive_ids = set()
        for line in archive_file.read_text().splitlines():
            line = line.strip()
            if line:
                parts = line.split()
                if len(parts) >= 2:
                    archive_ids.add(parts[1])
                elif len(parts) == 1:
                    archive_ids.add(parts[0])
        return archive_ids

    def render_ui(self):
        print("=== YouTube Specialist Downloader ===")
        self.mount_google_drive()

        # Unified Single URL Input
        self.url_input = widgets.Text(
            description="YouTube URL:",
            placeholder="Paste single video or playlist link here...",
            layout=widgets.Layout(width='90%')
        )

        self.folder = widgets.Text(
            description="Folder Name:",
            value="MyPlaylist",
            placeholder="Subfolder inside MyDrive/YouTubeDownloads",
            layout=widgets.Layout(width='90%')
        )

        self.dl_type = widgets.Dropdown(
            options=["Video + Audio", "Audio Only", "Video Only"],
            value="Video + Audio",
            description="Download Type:",
            layout=widgets.Layout(width='90%')
        )

        self.quality = widgets.Dropdown(
            options=["Best", "1080p", "720p", "480p", "360p"],
            value="Best",
            description="Max Quality:",
            layout=widgets.Layout(width='90%')
        )

        self.fast_resume = widgets.Checkbox(
            value=False,
            description="Fast Resume (Skip item checks & start immediately after last downloaded video)",
            indent=False,
            layout=widgets.Layout(width='90%')
        )

        self.start_btn = widgets.Button(description="Start Download", button_style='success', layout=widgets.Layout(width='90%'))
        self.start_btn.on_click(self.on_start)

        self.output = widgets.Output()

        display(
            self.url_input,
            self.folder,
            self.dl_type,
            self.quality,
            self.fast_resume,
            self.start_btn,
            self.output
        )

    def on_start(self, b):
        with self.output:
            clear_output()
            url = self.url_input.value.strip()
            folder_name = self.sanitize_name(self.folder.value.strip())
            dl_type = self.dl_type.value
            quality = self.quality.value
            use_fast_resume = self.fast_resume.value

            if not url:
                print("[ERROR] Please provide a valid YouTube URL.")
                return
            if not folder_name:
                print("[ERROR] Please specify a target folder name.")
                return

            target_dir = self.youtube_base_dir / folder_name
            target_dir.mkdir(parents=True, exist_ok=True)
            print(f"[INFO] Destination Directory: Google Drive -> MyDrive/YouTubeDownloads/{folder_name}")

            archive_file = target_dir / "download_archive.txt"

            # Automatic background archive file handling
            if archive_file.exists():
                print(f"[INFO] Auto-detected existing 'download_archive.txt' in Drive ({target_dir.name}). Resuming progress.")
            else:
                print(f"[INFO] Creating new 'download_archive.txt' in Drive ({target_dir.name}).")
                archive_file.write_text("")

            self.temp_dir.mkdir(parents=True, exist_ok=True)

            print("[INFO] Processing link and extracting metadata...")
            ydl_extract_opts = {'extract_flat': True, 'quiet': True}
            entries = []

            try:
                with YoutubeDL(ydl_extract_opts) as ydl:
                    info = ydl.extract_info(url, download=False)
                    if 'entries' in info:
                        entries = list(info['entries'])
                        print(f"[INFO] Detected Playlist containing {len(entries)} item(s).")
                    else:
                        entries = [info]
                        print("[INFO] Detected Single Video target.")
            except Exception as e:
                print(f"[ERROR] Failed to extract URL metadata: {e}")
                return

            # --- Fast Resume Logic ---
            if use_fast_resume:
                completed_ids = self.extract_downloaded_ids_from_archive(archive_file)
                if completed_ids:
                    last_completed_idx = -1
                    for idx, entry in enumerate(entries):
                        v_id = entry.get('id') or (entry.get('url', '').split('v=')[-1] if 'v=' in entry.get('url', '') else None)
                        if v_id and v_id in completed_ids:
                            last_completed_idx = idx

                    if last_completed_idx != -1:
                        skipped_count = last_completed_idx + 1
                        entries = entries[skipped_count:]
                        print(f"[FAST RESUME] Found last completed video at index {skipped_count}. Skipping {skipped_count} item(s) instantly!")
                    else:
                        print("[FAST RESUME] No matching completed IDs found in playlist. Starting from beginning.")
                else:
                    print("[FAST RESUME] Archive is empty. Starting from beginning.")

            format_str, postprocessors = self.get_format_config(dl_type, quality)

            total_remaining = len(entries)
            if total_remaining == 0:
                print("\n[DONE] All videos in this link are already completed!")
                return

            for i, entry in enumerate(entries, 1):
                vid_url = entry.get('url')
                if not vid_url and entry.get('id'):
                    vid_url = f"https://www.youtube.com/watch?v={entry.get('id')}"

                if not vid_url:
                    continue

                print(f"\n[{i}/{total_remaining}] Downloading ({dl_type} @ {quality}): {vid_url}")

                ydl_dl_opts = {
                    'format': format_str,
                    'outtmpl': str(self.temp_dir / '%(title).200s.%(ext)s'),
                    'download_archive': str(archive_file),
                    'postprocessors': postprocessors,
                    'quiet': False,
                    'noprogress': False
                }

                if dl_type == "Video + Audio":
                    ydl_dl_opts['merge_output_format'] = 'mp4'

                try:
                    with YoutubeDL(ydl_dl_opts) as ydl:
                        ydl.download([vid_url])
                except Exception as e:
                    print(f"[ERROR] Failed to download {vid_url}: {e}")

                transferred = self.transfer_folder_contents_to_drive(self.temp_dir, target_dir)
                if transferred:
                    print(f"[SUCCESS] Moved {len(transferred)} file(s) to Google Drive.")
                else:
                    print("[INFO] Item skipped (already in archive or failed).")

            print("\n[DONE] Execution completed successfully!")

In [ ]:
# Run UI
app = YouTubeSpecialistDL()
app.render_ui()